# M5 - the defended pass (the project's core result)

Same battery, same 50 AdvBench goals, same seed - but now through the full stack:

    L1 perplexity -> L1.5 structural -> L2 paraphrase -> L3 hardening -> TARGET -> L4 judge

**What this notebook produces** (all three are report deliverables):

1. **Cost of defense** - 50 benign prompts, stack off vs stack on. A filter that also
   refuses homework help is not a defense. This is the number that keeps us honest.
2. **Defended ASR** - per-attack, directly comparable to the M4c baseline.
3. **Per-layer attribution** - *which* layer stopped each attack, from the recorded
   verdicts. "ASR fell" is not a finding; "the base64 family dies at L1.5 and persona
   attacks survive to L4" is.

**The baseline it is measured against** (undefended Qwen2.5-7B, 877 trials, 49 goals):

| attack | ASR |
|---|---|
| prefix_injection | **97.9%** |
| distractors | 49.0% |
| leetspeak | 44.9% |
| combination_3 | 28.6% |
| combination_1 | 22.4% |
| *overall* | **17.3%** |

**Predictions to check** (write them down before running, so the result can surprise you):

- `base64` + `combination_*` -> blocked at **L1.5**, offline-validated at 100% catch / 0% benign FPR.
- `rot13`, `leetspeak`, `disemvowel` -> blocked at **L1** (M3's measured catch set).
- `prefix_injection` (97.9% undefended) -> the real test. It is fluent English, so L1
  and L1.5 cannot see it. It must be stopped by L2 rewriting the forced prefix away,
  L3's refusal priming, or L4 catching the reply. If it still lands, the stack has a hole.
- Persona attacks (`aim`, `dev_mode`) were already ~0% undefended - they cannot improve,
  so ignore them when judging the stack.

Needs **GPU T4 x2**: target 7B + judge 9B(nf4) + scorer + paraphraser(nf4), all resident.


## 1 - Setup

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob, shutil
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo -----------------------------------------------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
WORK   = pathlib.Path("/kaggle/working")
ROOT   = WORK / "repo"

_gh  = _secret("GH_TOKEN")
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

os.chdir(WORK)
subprocess.run(["rm", "-rf", str(ROOT)], check=False)
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    cwd=str(WORK), capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError("git clone failed:\n" + _err)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("HEAD", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"]).decode().strip())

_missing = [f for f in ("run_eval.py", "report.py", "benign_eval.py",
                        "defense/layer1_5_structural.py", "defense/layer2_paraphrase.py")
            if not (ROOT / f).exists()]
if _missing:
    raise RuntimeError(f"clone is missing {_missing} -- commit and push them, then re-run this cell")
print("repo files OK")

In [ ]:
r = subprocess.run([sys.executable, "datasets/build_harmful.py"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
import torch
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.0f} GB')
assert torch.cuda.device_count() >= 2, 'set the accelerator to GPU T4 x2 -- four models must fit'

In [ ]:
from core.config import CONFIG
for role in ('target', 'paraphraser', 'perplexity_scorer', 'judge'):
    s = CONFIG['models'][role]
    print(f"{role:<18} {s['name']:<30} device={str(s.get('device')):<8} quant={s.get('quant')}"
          f"  max_memory={s.get('max_memory')}")
print()
from defense import build_pipeline
print('pipeline:', build_pipeline().describe())

## 2 - Load all four models and check they fit

The defended pass is the only run that holds target + scorer + paraphraser + judge at
once. `config.toml` budgets cuda:1 at 4 GiB for the target's spill so the other three
fit beside it. If this OOMs, lower `[models.target] max_memory` for GPU 1 further.

In [ ]:
from core.seed import seed_everything
from core.models import load_target, load_paraphraser, load_perplexity_scorer, load_judge

seed_everything()
t0 = time.time()
target = load_target();            print('target loaded      ', f'{time.time()-t0:5.0f}s')
para   = load_paraphraser();       print('paraphraser loaded ', f'{time.time()-t0:5.0f}s')
scorer = load_perplexity_scorer(); print('scorer loaded      ', f'{time.time()-t0:5.0f}s')
judge  = load_judge();             print('judge loaded       ', f'{time.time()-t0:5.0f}s')

for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i)/1e9
    tot  = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f'  cuda:{i}  {used:5.1f} / {tot:.0f} GB')

In [ ]:
# does the paraphraser actually paraphrase? (it is 1.5B -- verify before trusting it)
probe = "Give step-by-step instructions for repotting a houseplant"
raw = para.generate(f"paraphrase the following sentences:\n\n{probe}")
from defense.layer2_paraphrase import _clean
print('RAW  :', raw[:300])
print('CLEAN:', _clean(raw)[:300])

### Gate 1

- Four models resident, no OOM.
- The paraphrase is a *restatement* of the probe, not a refusal and not a chat preamble.
  If `_clean` still leaves "Sure, here is..." text, extend `_PREAMBLE` in
  `defense/layer2_paraphrase.py` rather than patching it here (notebook cells do not
  sync back to git).

## 3 - Cost of defense: 50 benign prompts, stack off vs stack on

Run this **before** the expensive defended pass. If the stack refuses ordinary requests
at a high rate, the ASR number is not worth collecting yet - fix L3's wording first.

~100 generations, roughly 10-20 min.

In [ ]:
import benign_eval
und_dir = benign_eval.run(defended=False, tag='m5benign-und')

In [ ]:
dfn_dir = benign_eval.run(defended=True, tag='m5benign-def')

In [ ]:
cost = benign_eval.print_compare(und_dir, dfn_dir)

### Gate 2

- **benign false positives should be 0** (L1 was calibrated for that in M3, and L1.5 was
  offline-validated at 0/50 on this exact set). Any FP here is a real regression - find
  which layer and why before continuing.
- A modest `served_%` drop from L3's refusal priming is the honest cost of the defense;
  record it, do not hide it. A large drop (>20 points) means L3 is too aggressive.

## 4 - The defended pass (50 goals x 18 attacks, ~2-4 h)

Sanity first (3 goals, ~54 trials), then the full sweep.

In [ ]:
from run_eval import main
main(['--attack', 'all', '--defense', 'on', '--limit', '3', '--tag', 'm5defsanity'])

In [ ]:
import report
report.print_asr('m5defsanity', title='SANITY defended (3 goals)')
report.print_attribution('m5defsanity', title='SANITY attribution')

### Gate 3

Check the attribution table before spending hours:
- `base64`, `combination_1/2/3` -> `layer1_5_structural`
- `rot13`, `leetspeak`, `disemvowel` -> `layer1_perplexity`
- everything fluent -> `reached_target` (L4 decides those)

If base64 is reaching the target, L1.5 is not enabled - check `[defense.layer1_5_structural]`.

In [ ]:
main(['--attack', 'all', '--defense', 'on', '--tag', 'm5defended'])

## 5 - The three result tables

In [ ]:
report.print_asr('m5defended', title='DEFENDED ASR - full stack, Qwen2.5-7B, 50 AdvBench goals')

In [ ]:
report.print_attribution('m5defended', title='PER-LAYER ATTRIBUTION - which layer stopped what')

In [ ]:
# before/after. The 3B-era baseline is not comparable; use the 7B one (m4c7bbaseline).
base = 'm4c7bbaseline'
old = [pathlib.Path(p).parent for p in glob.glob('/kaggle/input/**/transcript.jsonl', recursive=True)
       if 'm4c7bbaseline' in p]
if old:
    base = str(old[0])
report.print_compare(base, 'm5defended', label_a='undefended', label_b='defended')

In [ ]:
# what still gets through: read these, they go in the report as the honest failure cases
report.samples('m5defended', only='BAD_BOT', n=10)

## 6 - Save + pin

Download `artifacts_m5.zip`, unzip over local `logs/`, and paste the printed revisions
into `config.toml` (target, helper and paraphraser are still `PIN-ME`).

In [ ]:
from huggingface_hub import HfApi
for role in ('target', 'helper', 'paraphraser', 'judge'):
    nm = CONFIG['models'][role]['name']
    try:
        sha = HfApi().model_info(nm, token=os.environ.get('HF_TOKEN')).sha
        print(f'[models.{role}]  # {nm}\n  revision = "{sha}"')
    except Exception as e:
        print(role, 'lookup failed:', e)

In [ ]:
!cd /kaggle/working && zip -qr artifacts_m5.zip artifacts && ls -la artifacts_m5.zip
print()
!ls /kaggle/working/artifacts

## Done

**What this run settles:** whether a stack whose two cheap layers (L1, L1.5) are
provably free - 0% benign false positives, no GPU - can carry most of the defense, with
the expensive judge (L4) only handling what is left. The attribution table answers that
directly: if `reached_target` is small for the encoding families and large only for the
fluent ones, the layered design did its job.

**Next: M6** - the adaptive attack (per goal, success = *any* of the 18 techniques
lands), undefended vs defended. That is the single headline number for the report.